# Planners-6-Domains-Csharp — Jumeau C# : planificateur STRIPS from-scratch

> **Jumeau C# (.NET 9)** de [`Planners-6-Domains`](Planners-6-Domains.ipynb). Le notebook Python invoque le vrai moteur `unified-planning` (solveur `pyperplan`) pour résoudre des domaines PDDL classiques. **Ce jumeau prend le parti inverse (Prong B du marathon #4956)** : plutôt que d'appeler un solveur, on **reconstruit un planificateur STRIPS from-scratch** en C# pur (BCL .NET 9, zéro NuGet), puis on l'exécute sur les mêmes domaines canoniques (Block World, Tours de Hanoï, Gripper). L'objectif pédagogique est de rendre **visible la mécanique interne** que `pyperplan` cache : représentation de l'état, recherche avant (forward search), application des effets add/delete.

**Source Python** : [`Planners-6-Domains.ipynb`](Planners-6-Domains.ipynb) — domaines PDDL + `unified-planning.shortcuts.OneshotPlanner(name='pyperplan')`.

## Objectifs d'apprentissage

1. **Modéliser** un domaine STRIPS en C# : atomes, préconditions, effets (add/delete lists)
2. **Implémenter** un planificateur par **recherche avant** (BFS sur l'espace d'états) avec détection de cycles
3. **Reproduire** les domaines classiques (Block World, Hanoï, Gripper) et comparer les plans obtenus
4. **Mesurer** le coût de la recherche (nœuds explorés, longueur du plan) pour comprendre les enjeux d'explosion combinatoire

### Prérequis

- Concepts STRIPS (préconditions, effets) — voir [`Planners-2-PDDL-Basics`](Planners-2-PDDL-Basics.ipynb)
- Recherche en espace d'états (BFS) — voir [`Planners-3-State-Space`](Planners-3-State-Space.ipynb)
- .NET 9 + kernel `csharp` (kernel `.net-csharp`)

### Durée estimée : 45 minutes

```mermaid
flowchart LR
    A["Modèle STRIPS<br/>(Atom, Action, State)"] --> B["Planificateur BFS<br/>forward search"]
    B --> C["Block World<br/>tour + reverse"]
    B --> D["Tours de Hanoï<br/>3 disques"]
    B --> E["Gripper<br/>2 balles, 2 pinces"]
    C --> F["Analyse<br/>longueur / nœuds"]
    D --> F
    E --> F
```


## 1. Modèle STRIPS en C# pur

Un état STRIPS est un **ensemble d'atomes booléens grounded** (ex. `on(a,b)`, `clear(a)`, `handempty`). Une **action** a un nom, des paramètres (déjà instanciés ici), une **précondition** (ensemble d'atomes qui doivent tous être vrais) et un **effet** décomposé en listes `add` / `del` (ce que l'action rend vrai / faux).

On représente chaque atome par une simple chaîne normalisée — pas de parsing PDDL, on reste au plus près de la sémantique.

In [1]:

// Modèle STRIPS from-scratch (BCL .NET 9, 0 NuGet)
using System;
using System.Collections.Generic;
using System.Linq;

// Un atome grounded : "on(a,b)", "clear(c)", "handempty", "at(b1,roomA)".
// Représenté par une chaîne normalisée (trim + lower des args).
public readonly struct Atom : IEquatable<Atom>
{
    public readonly string Text;
    public Atom(string text) { Text = text.Trim(); }
    public bool Equals(Atom other) => Text == other.Text;
    public override bool Equals(object? o) => o is Atom a && Equals(a);
    public override int GetHashCode() => Text.GetHashCode();
    public override string ToString() => Text;
    public static bool operator ==(Atom a, Atom b) => a.Equals(b);
    public static bool operator !=(Atom a, Atom b) => !a.Equals(b);
}

// Une action STRIPS grounded : précondition (conjonction d'atomes) + effet add/del.
public sealed class Action
{
    public string Name { get; }
    public HashSet<Atom> Pre { get; }
    public HashSet<Atom> Add { get; }
    public HashSet<Atom> Del { get; }
    public Action(string name, IEnumerable<string>? pre = null,
                  IEnumerable<string>? add = null, IEnumerable<string>? del = null)
    {
        Name = name;
        Pre = (pre ?? Array.Empty<string>()).Select(s => new Atom(s)).ToHashSet();
        Add = (add ?? Array.Empty<string>()).Select(s => new Atom(s)).ToHashSet();
        Del = (del ?? Array.Empty<string>()).Select(s => new Atom(s)).ToHashSet();
    }
    public override string ToString() => Name;
}

// État : ensemble d'atomes. Immuable par copie à chaque transition.
public sealed class State : IEquatable<State>
{
    public HashSet<Atom> Facts { get; }
    public State(IEnumerable<string> facts) { Facts = facts.Select(s => new Atom(s)).ToHashSet(); }
    public State(HashSet<Atom> facts) { Facts = facts; }
    public bool Contains(Atom a) => Facts.Contains(a);
    public State Clone() => new State(new HashSet<Atom>(Facts));

    // Une action est applicable ssi toutes ses préconditions sont satisfaites.
    public bool CanApply(Action a) => a.Pre.All(p => Facts.Contains(p));

    // Appliquer : on retire Del puis on ajoute Add (ordre add-after-del, convention STRIPS).
    public State Apply(Action a)
    {
        var next = Clone();
        foreach (var d in a.Del) next.Facts.Remove(d);
        foreach (var x in a.Add) next.Facts.Add(x);
        return next;
    }
    public bool Equals(State? other) =>
        other is not null && Facts.SetEquals(other.Facts);
    public override bool Equals(object? o) => o is State s && Equals(s);
    public override int GetHashCode()
    {
        // Ordre-indépendant : XOR des hashes des atomes (suffisant pour BFS small-state).
        int h = 0;
        foreach (var a in Facts) h ^= a.GetHashCode();
        return h;
    }
}

Console.WriteLine("Modèle STRIPS chargé : Atom, Action (pre/add/del), State (immuable).");


Modèle STRIPS chargé : Atom, Action (pre/add/del), State (immuable).



(14,39): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(28,51): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(29,38): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(29,71): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(59,29): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(61,39): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



## 2. Planificateur par recherche avant (BFS)

Le moteur le plus simple : une **recherche en largeur** (BFS) dans l'espace d'états. À chaque nœud, on instancie toutes les actions applicables, on génère les états successeurs, et on garde la trace du chemin parcouru. La BFS garantit un plan **de longueur minimale** (en nombre d'actions) — c'est exactement ce que l'on veut pour Block World. Deux raffinements essentiels :

- **Détection de cycles** : on mémorise les états déjà visités (via le hash ordre-indépendant) pour éviter de boucler.
- **Test de but** : le but est un ensemble d'atomes qui doivent **tous** être présents dans l'état courant.

In [2]:

// Résultat d'une recherche : plan trouvé (séquence d'actions) + métriques.
public sealed class PlanResult
{
    public List<Action> Actions { get; set; } = new();
    public int NodesExpanded { get; set; }
    public int NodesGenerated { get; set; }
    public bool Solved { get; set; }
}

public static class StripsPlanner
{
    // BFS forward-search. `goal` = atomes qui doivent tous être vrais à l'arrivée.
    public static PlanResult Solve(State init, List<Action> actions, HashSet<Atom> goal,
                                   int maxNodes = 200_000)
    {
        var result = new PlanResult();

        // File de (état, chemin d'actions pour l'atteindre).
        var queue = new Queue<(State state, List<Action> path)>();
        queue.Enqueue((init, new List<Action>()));

        var visited = new HashSet<State> { init };

        while (queue.Count > 0)
        {
            var (cur, path) = queue.Dequeue();
            result.NodesExpanded++;

            // Test de but : tous les atomes du but sont-ils présents ?
            if (goal.All(g => cur.Contains(g)))
            {
                result.Actions = path;
                result.Solved = true;
                return result;
            }

            if (result.NodesExpanded > maxNodes)
                return result;  // explosion combinatoire — on abandonne proprement

            // Successeurs : chaque action applicable produit un nouvel état.
            foreach (var a in actions)
            {
                if (!cur.CanApply(a)) continue;
                var next = cur.Apply(a);
                if (visited.Contains(next)) continue;   // anti-cycle
                visited.Add(next);
                var newPath = new List<Action>(path) { a };
                queue.Enqueue((next, newPath));
                result.NodesGenerated++;
            }
        }
        return result;  // Solved = false : but injoignable
    }

    // Affiche un plan + métriques (BFS = plan minimal en nb d'actions).
    public static void PrintPlan(PlanResult r)
    {
        if (!r.Solved)
        {
            Console.WriteLine("AUCUN PLAN TROUVÉ (but injoignable ou explosion combinatoire).");
            Console.WriteLine($"  Nœuds explorés : {r.NodesExpanded}");
            return;
        }
        Console.WriteLine($"Solution trouvée ! Longueur du plan : {r.Actions.Count} action(s)");
        Console.WriteLine(new string('=', 50));
        for (int i = 0; i < r.Actions.Count; i++)
            Console.WriteLine($"  {i+1}. {r.Actions[i]}");
        Console.WriteLine(new string('=', 50));
        Console.WriteLine($"Nœuds explorés (expanded) : {r.NodesExpanded}");
        Console.WriteLine($"Nœuds générés (generated) : {r.NodesGenerated}");
    }
}

Console.WriteLine("Planificateur BFS chargé : StripsPlanner.Solve (forward search + anti-cycle).");


Planificateur BFS chargé : StripsPlanner.Solve (forward search + anti-cycle).


## 3. Block World — construire une tour (c ← b ← a)

On reprend **exactement** le problème `blocks-tower` du notebook Python : trois blocs `a, b, c` posés sur la table, bras vide ; but `on(a,b)` et `on(b,c)` (tour `c` portant `b` portant `a`). Les actions STRIPS sont les classiques `pick-up`, `put-down`, `stack`, `unstack`.

Le moteur `pyperplan` (Python) trouve un plan en **4 actions**. Vérifions que notre BFS from-scratch retrouve la même longueur minimale.

In [3]:

// --- Domaine Block World : actions grounded pour 3 blocs a, b, c ---
// Une fois instanciées, les actions sont des opérateurs concrets ; le planificateur
// les essaie toutes à chaque nœud (forward search).

var blocksActions = new List<Action>();

// pick-up(x) : prendre un bloc sur la table. Pre: clear(x), ontable(x), handempty.
foreach (var x in new[] { "a", "b", "c" })
{
    blocksActions.Add(new Action($"pick-up({x})",
        pre: new[] { $"clear({x})", $"ontable({x})", "handempty" },
        add: new[] { $"holding({x})" },
        del: new[] { $"clear({x})", $"ontable({x})", "handempty" }));

    // put-down(x) : poser le bloc tenu sur la table.
    blocksActions.Add(new Action($"put-down({x})",
        pre: new[] { $"holding({x})" },
        add: new[] { $"ontable({x})", $"clear({x})", "handempty" },
        del: new[] { $"holding({x})" }));

    // stack(x, y) : poser x sur y (y doit être clear).
    foreach (var y in new[] { "a", "b", "c" })
        if (x != y)
            blocksActions.Add(new Action($"stack({x},{y})",
                pre: new[] { $"holding({x})", $"clear({y})" },
                add: new[] { $"on({x},{y})", $"clear({x})", "handempty" },
                del: new[] { $"holding({x})", $"clear({y})" }));

    // unstack(x, y) : prendre x qui est sur y.
    foreach (var y in new[] { "a", "b", "c" })
        if (x != y)
            blocksActions.Add(new Action($"unstack({x},{y})",
                pre: new[] { $"on({x},{y})", $"clear({x})", "handempty" },
                add: new[] { $"holding({x})", $"clear({y})" },
                del: new[] { $"on({x},{y})", $"clear({x})", "handempty" }));
}

// État initial : a, b, c sur la table, tous clear, bras vide.
var bwInit = new State(new[] { "ontable(a)", "ontable(b)", "ontable(c)",
                                "clear(a)", "clear(b)", "clear(c)", "handempty" });

// But : tour c <- b <- a  (a sur b, b sur c).
var bwGoal = new HashSet<Atom> { new("on(a,b)"), new("on(b,c)") };

Console.WriteLine("Problème 'blocks-tower' : construire la tour c <- b <- a");
Console.WriteLine("État initial : a, b, c sur la table\n");
var bwTower = StripsPlanner.Solve(bwInit, blocksActions, bwGoal);
StripsPlanner.PrintPlan(bwTower);


Problème 'blocks-tower' : construire la tour c <- b <- a


État initial : a, b, c sur la table



Solution trouvée ! Longueur du plan : 4 action(s)


  1. pick-up(b)


  2. stack(b,c)


  3. pick-up(a)


  4. stack(a,b)


Nœuds explorés (expanded) : 20


Nœuds générés (generated) : 21


### 3.1 Inverser deux tours (problème `blocks-reverse`)

Le second problème du notebook Python : deux tours initiales `table <- c <- d` et `table <- a <- b`, à inverser en `table <- d <- c` et `table <- b <- a`. On étend le domaine à 4 blocs. La BFS reste exacte (plan minimal), mais le nombre de nœuds explorés explose — c'est l'**explosion combinatoire** que le notebook Python mentionne en §5.2, rendue ici tangible par notre compteur `NodesExpanded`.

In [4]:

// --- Block World étendu à 4 blocs a, b, c, d ---
var all4 = new[] { "a", "b", "c", "d" };
var blocksActions4 = new List<Action>();
foreach (var x in all4)
{
    blocksActions4.Add(new Action($"pick-up({x})",
        pre: new[] { $"clear({x})", $"ontable({x})", "handempty" },
        add: new[] { $"holding({x})" },
        del: new[] { $"clear({x})", $"ontable({x})", "handempty" }));
    blocksActions4.Add(new Action($"put-down({x})",
        pre: new[] { $"holding({x})" },
        add: new[] { $"ontable({x})", $"clear({x})", "handempty" },
        del: new[] { $"holding({x})" }));
    foreach (var y in all4)
        if (x != y)
        {
            blocksActions4.Add(new Action($"stack({x},{y})",
                pre: new[] { $"holding({x})", $"clear({y})" },
                add: new[] { $"on({x},{y})", $"clear({x})", "handempty" },
                del: new[] { $"holding({x})", $"clear({y})" }));
            blocksActions4.Add(new Action($"unstack({x},{y})",
                pre: new[] { $"on({x},{y})", $"clear({x})", "handempty" },
                add: new[] { $"holding({x})", $"clear({y})" },
                del: new[] { $"on({x},{y})", $"clear({x})", "handempty" }));
        }
}

// État initial : table <- c <- d  ET  table <- a <- b.
var bw4Init = new State(new[] {
    "ontable(c)", "on(d,c)", "clear(d)",
    "ontable(a)", "on(b,a)", "clear(b)", "handempty" });

// But : inverser -> table <- d <- c  ET  table <- b <- a.
var bw4Goal = new HashSet<Atom> {
    new("ontable(d)"), new("on(c,d)"),
    new("ontable(b)"), new("on(a,b)") };

Console.WriteLine("Problème 'blocks-reverse' : inverser deux tours (4 blocs)\n");
var bwReverse = StripsPlanner.Solve(bw4Init, blocksActions4, bw4Goal);
StripsPlanner.PrintPlan(bwReverse);


Problème 'blocks-reverse' : inverser deux tours (4 blocs)



Solution trouvée ! Longueur du plan : 8 action(s)


  1. unstack(b,a)


  2. put-down(b)


  3. pick-up(a)


...

  6. put-down(d)


  7. pick-up(c)


  8. stack(c,d)


Nœuds explorés (expanded) : 85


Nœuds générés (generated) : 105


## 4. Tours de Hanoï (domaine `hanoi`)

Le problème le plus célèbre de planification classique. Trois disques `d1 < d2 < d3` (du plus petit au plus grand), trois piquets `p1, p2, p3`. État initial : tous sur `p1` (`d3` en bas). But : tous sur `p3`. La règle STRIPS encode « un disque ne peut être posé que sur un disque plus grand ou sur un piquet vide ».

La solution optimale en 3 disques fait **7 coups** (formule $2^n - 1$). Notons qu'en STRIPS pur, il faut **modéliser la contrainte de taille** explicitement — ici via le prédicat `clear` (rien au-dessus) et en n'autorisant `disk-on(X,Y)` que pour `Y` plus grand que `X` (les actions grounded sont générées uniquement pour les paires licites).

In [5]:

// --- Tours de Hanoï : 3 disques, 3 piquets ---
// Convention : d1 < d2 < d3 (d3 = plus grand). Un disque ne va que sur un plus grand.
var pegs = new[] { "p1", "p2", "p3" };
var disks = new[] { "d1", "d2", "d3" };   // d1 = petit, d3 = grand
int Rank(string d) => Array.IndexOf(disks, d);

var hanoiActions = new List<Action>();
// move(x, from, to) : déplacer le disque x (au sommet de `from`) vers `to`.
// `from` parcourt piquets ET disques (un disque peut reposer sur un disque).
// CONTRAINTE DE TAILLE : on ne génère l'action que pour `to` licite (piquet OU disque
// plus grand que x). C'est ce qui empêche de poser un grand disque sur un petit.
var allSupports = pegs.Concat(disks).ToList();
foreach (var x in disks)
    foreach (var from in allSupports)
    {
        var legalDests = new List<string>(pegs);
        foreach (var y in disks)
            if (Rank(y) > Rank(x)) legalDests.Add(y);   // y plus grand -> x peut aller dessus
        foreach (var to in legalDests)
            if (from != to)
                hanoiActions.Add(new Action($"move({x},{from}->{to})",
                    pre: new[] { $"on({x},{from})", $"clear({x})", $"clear({to})" },
                    add: new[] { $"on({x},{to})", $"clear({from})" },
                    del: new[] { $"on({x},{from})", $"clear({to})" }));
    }

// État initial : d3 sur p1, d2 sur d3, d1 sur d2 (d1 au sommet, clear).
var hanoiInit = new State(new[] {
    "on(d3,p1)", "on(d2,d3)", "on(d1,d2)", "clear(d1)",
    "clear(p2)", "clear(p3)" });
// But : tout sur p3 (d3 en bas, d2 au milieu, d1 au sommet).
var hanoiGoal = new HashSet<Atom> {
    new("on(d3,p3)"), new("on(d2,d3)"), new("on(d1,d2)") };

Console.WriteLine("Tours de Hanoï : 3 disques de p1 vers p3\n");
var hanoi = StripsPlanner.Solve(hanoiInit, hanoiActions, hanoiGoal);
StripsPlanner.PrintPlan(hanoi);
Console.WriteLine($"\nThéorie : 2^3 - 1 = {Math.Pow(2,3)-1} coups optimaux attendus.");


Tours de Hanoï : 3 disques de p1 vers p3



Solution trouvée ! Longueur du plan : 7 action(s)


  1. move(d1,d2->p3)


  2. move(d2,d3->p2)


  3. move(d1,p3->d2)


...

  6. move(d2,p2->d3)


  7. move(d1,p1->d2)


Nœuds explorés (expanded) : 25


Nœuds générés (generated) : 26



Théorie : 2^3 - 1 = 7 coups optimaux attendus.


### Interprétation : Hanoï et l'explosion combinatoire

La formule $2^n - 1$ donne la longueur du plan optimal, mais le **coût de la recherche** est bien pire. Notre BFS explore tous les états intermédiaires atteignables ; pour Hanoï à $n$ disques, l'espace d'états compte $3^n$ configurations légales. À $n = 3$, c'est trivial (27 états) ; à $n = 10$, on dépasse **59 000 états** — et un solveur PDDL comme `pyperplan` s'appuie justement sur des **heuristiques** (relaxation delete-free) pour ne pas tout explorer. C'est toute la différence entre notre BFS pédagogique (exact mais naïf) et un moteur de production : le nôtre *montre* la mécanique, `pyperplan` *l'optimise*.

## 5. Domaine Gripper (robot à deux pinces)

Le domaine `gripper` du notebook Python : un robot dans une pièce, doté de deux pinces (`g1`, `g2`), doit déplacer des balles d'une pièce (`roomA`) à l'autre (`roomB`). Trois actions : `move`, `pick`, `drop`. L'intérêt pédagogique est la **parallélisation potentielle** : avec deux pinces, le robot peut porter deux balles d'un coup, ce qui réduit le nombre de `move` (coûteux). Notre BFS retrouve le plan minimal — et on voit directement l'arbitrage *nombre de moves* vs *nombre de picks/drops*.

In [6]:

// --- Domaine Gripper : 2 balles, 2 pinces, 2 pièces ---
var rooms = new[] { "roomA", "roomB" };
var balls = new[] { "b1", "b2" };
var grippers = new[] { "g1", "g2" };

var gripperActions = new List<Action>();
// move(from, to) : déplacer le robot.
foreach (var from in rooms)
    foreach (var to in rooms)
        if (from != to)
            gripperActions.Add(new Action($"move({from}->{to})",
                pre: new[] { $"at-robby({from})" },
                add: new[] { $"at-robby({to})" },
                del: new[] { $"at-robby({from})" }));

// pick(ball, room, gripper) / drop(ball, room, gripper).
foreach (var b in balls)
    foreach (var r in rooms)
        foreach (var g in grippers)
        {
            gripperActions.Add(new Action($"pick({b},{r},{g})",
                pre: new[] { $"at({b},{r})", $"at-robby({r})", $"free({g})" },
                add: new[] { $"carry({b},{g})" },
                del: new[] { $"at({b},{r})", $"free({g})" }));
            gripperActions.Add(new Action($"drop({b},{r},{g})",
                pre: new[] { $"carry({b},{g})", $"at-robby({r})" },
                add: new[] { $"at({b},{r})", $"free({g})" },
                del: new[] { $"carry({b},{g})" }));
        }

// État initial : robot + balles en roomA, deux pinces libres.
var gripInit = new State(new[] {
    "at-robby(roomA)", "at(b1,roomA)", "at(b2,roomA)", "free(g1)", "free(g2)" });
// But : les deux balles en roomB.
var gripGoal = new HashSet<Atom> { new("at(b1,roomB)"), new("at(b2,roomB)") };

Console.WriteLine("Gripper : transporter 2 balles de roomA vers roomB (2 pinces)\n");
var grip = StripsPlanner.Solve(gripInit, gripperActions, gripGoal);
StripsPlanner.PrintPlan(grip);


Gripper : transporter 2 balles de roomA vers roomB (2 pinces)



Solution trouvée ! Longueur du plan : 5 action(s)


  1. pick(b1,roomA,g1)


  2. pick(b2,roomA,g2)


  3. move(roomA->roomB)


  4. drop(b1,roomB,g1)


  5. drop(b2,roomB,g2)


Nœuds explorés (expanded) : 25


Nœuds générés (generated) : 26


## 6. Analyse comparative des domaines

On récapitule les quatre problèmes résolus par notre BFS from-scratch. Deux métriques : **longueur du plan** (optimale en BFS) et **nœuds explorés** (le coût réel de la recherche). C'est ce second chiffre qui révèle l'enjeu : Block World à 4 blocs et Hanoï à 3 disques sont petits, mais la croissance est exponentielle — d'où le besoin, dans un solveur industriel, d'heuristiques.

In [7]:

// --- Tableau récapitulatif des 4 plans ---
var results = new (string Domain, int PlanLen, int Expanded, bool Solved)[]
{
    ("Block World (tour 3 blocs)", bwTower.Actions.Count, bwTower.NodesExpanded, bwTower.Solved),
    ("Block World (reverse, 4 blocs)", bwReverse.Actions.Count, bwReverse.NodesExpanded, bwReverse.Solved),
    ("Tours de Hanoï (3 disques)", hanoi.Actions.Count, hanoi.NodesExpanded, hanoi.Solved),
    ("Gripper (2 balles, 2 pinces)", grip.Actions.Count, grip.NodesExpanded, grip.Solved),
};

Console.WriteLine($"{"Domaine",-32} {"Plan",6} {"Nœuds explorés",16} {"Statut",10}");
Console.WriteLine(new string('-', 66));
foreach (var r in results)
    Console.WriteLine($"{r.Domain,-32} {r.PlanLen,6} {r.Expanded,16} {(r.Solved ? "RÉSOLU" : "ÉCHEC"),10}");
Console.WriteLine();
Console.WriteLine("Lecture : le plan est optimal (BFS), mais 'Nœuds explorés' croît");
Console.WriteLine("exponentiellement avec la taille du domaine — d'où les heuristiques de pyperplan.");


Domaine                            Plan   Nœuds explorés     Statut


------------------------------------------------------------------


Block World (tour 3 blocs)            4               20     RÉSOLU


Block World (reverse, 4 blocs)        8               85     RÉSOLU


Tours de Hanoï (3 disques)            7               25     RÉSOLU


Gripper (2 balles, 2 pinces)          5               25     RÉSOLU


Lecture : le plan est optimal (BFS), mais 'Nœuds explorés' croît


exponentiellement avec la taille du domaine — d'où les heuristiques de pyperplan.


### Comparaison avec le moteur `pyperplan` (notebook Python)

| Domaine | Plan BFS from-scratch (C#) | Plan `pyperplan` (Python) | Statut |
|---------|----------------------------|---------------------------|--------|
| Block World tour (3 blocs) | minimal (BFS) | 4 actions (rapporté) | Concordant |
| Block World reverse (4 blocs) | minimal (BFS) | — | — |
| Tours de Hanoï (3 disques) | $2^3-1 = 7$ | idem (classique) | Concordant |
| Gripper (2 balles) | minimal (BFS) | — | — |

Les longueurs concordent parce que **BFS garantit l'optimalité en nombre d'actions**. La différence est ailleurs : `pyperplan` explore **bien moins de nœuds** grâce à sa relaxation heuristique (il estime la distance au but sans les effets `del`), tandis que notre BFS naïf paie le coût intégral de l'explosion combinatoire. C'est précisément la leçon du notebook Python §5.2 : la modélisation déclare *le problème*, l'heuristique fait *toute la différence de performance*.

## Exercices

Trois exercices pour aller plus loin. Chacun est laissé en **stub** — votre travail est de compléter le code. Les exercices s'exécutent sans erreur même non complétés (convention C.1 : stub sans `throw`).

In [8]:

// Exercice 1 : ajouter un domaine — "Tirelire" (deposit, withdraw).
// Objectif : modéliser un compteur d'argent. Prédicat balance(N) pour N dans {0..5}.
// Actions : deposit (balance(N) -> balance(N+1)), withdraw (balance(N) -> balance(N-1)).
// But : passer de balance(0) à balance(3).
// Indice : générez les actions dans une boucle pour chaque N licite (0..4 pour deposit).
// Étape 1 : créer la liste d'actions.
// Étape 2 : définir l'état initial et le but.
// Étape 3 : appeler StripsPlanner.Solve et afficher.

// TODO étudiant : compléter ci-dessous.
var piggyActions = new List<Action>();   // à remplir
var piggyInit = new State(Array.Empty<string>());   // à remplir : balance(0)
var piggyGoal = new HashSet<Atom>();   // à remplir : balance(3)

var piggy = StripsPlanner.Solve(piggyInit, piggyActions, piggyGoal);
Console.WriteLine("Exercice 1 (Tirelire) — à compléter par l'étudiant.");


Exercice 1 (Tirelire) — à compléter par l'étudiant.


In [9]:

// Exercice 2 : mesurer l'explosion combinatoire sur Hanoï.
// Objectif : ajouter un 4e disque (d4, plus grand que d3) et mesurer NodesExpanded.
// Indice : la solution optimale fait 2^4 - 1 = 15 coups ; comptez les nœuds explorés.
// Question d'analyse : par quel facteur NodesExpanded augmente-t-il vs 3 disques ?
// Étape 1 : étendre la liste `disks` à d1..d4 et régénérer hanoiActions.
// Étape 2 : reconstruire l'état initial (d4 en bas de p1) et le but.
// Étape 3 : Solve + PrintPlan, noter NodesExpanded.

Console.WriteLine("Exercice 2 (Hanoï 4 disques) — à compléter par l'étudiant.");
// TODO étudiant : dupliquer le bloc Hanoï ci-dessus avec disks = { d1, d2, d3, d4 }.


Exercice 2 (Hanoï 4 disques) — à compléter par l'étudiant.


In [10]:

// Exercice 3 : amélioration — remplacer BFS par A* avec une heuristique.
// Objectif : ajouter une heuristique "nombre d'atomes du but non satisfaits"
// (heuristique naïve, non admissible mais informative) et passer à une priority queue.
// Indice : Priority Queue existe en BCL (System.Collections.Generic.PriorityQueue<T, double>).
// Étape 1 : écrire une fonction h(State) = nb d'atomes du goal absents.
// Étape 2 : remplacer la Queue par une PriorityQueue<(State, path), double>, clé = path.Count + h(state).
// Étape 3 : comparer NodesExpanded (BFS vs A* naïf) sur Block World reverse.

Console.WriteLine("Exercice 3 (A* avec heuristique naïve) — à compléter par l'étudiant.");
// TODO étudiant : implémenter StripsPlanner.AStar et comparer.


Exercice 3 (A* avec heuristique naïve) — à compléter par l'étudiant.


## Conclusion

Ce jumeau C# reconstruit **à la main** ce que `unified-planning` (notebook Python) délègue à un solveur externe. Les deux approches sont **complémentaires** et servent des apprentissages différents :

- **Le notebook Python** apprend à **modéliser** en PDDL et à appeler un moteur industriel — compétence pratique (on ne réécrit pas un solveur en production).
- **Ce jumeau C#** apprend **comment fonctionne** un planificateur : représentation d'état, recherche avant, application d'effets, explosion combinatoire. C'est la mécanique interne rendue visible.

### Ce que vous avez appris

1. Un **état STRIPS** est un ensemble d'atomes ; une **action** est précondition + effets add/del.
2. La **recherche avant (BFS)** garantit un plan minimal mais paie l'explosion combinatoire.
3. Les **heuristiques** (relaxation delete-free chez `pyperplan`) sont ce qui rend la planification tractable sur de vrais domaines — notre BFS naïf atteint vite ses limites.
4. Les **domaines classiques** (Block World, Hanoï, Gripper) partagent la même structure (préconditions/effectes) mais exposent des défis de modélisation distincts (taille relative, parallélisme).

### Prochaines étapes

- [`Planners-7-OR-Tools`](Planners-7-OR-Tools.ipynb) : bascule vers la **programmation par contraintes** (CP-SAT) pour l'ordonnancement — un autre paradigme de planification.
- [`Planners-8-Temporal`](../03-Advanced/Planners-8-Temporal.ipynb) : planification temporelle (PDDL 2.1, actions duratives).
- Pour la version "moteur réel", revenir au notebook Python [`Planners-6-Domains`](Planners-6-Domains.ipynb) et son appel à `OneshotPlanner(name='pyperplan')`.

## Ressources

- Fikes, R. E., & Nilsson, N. J. (1971) — « STRIPS: A New Approach to the Application of Theorem Proving to Problem Solving », *Artificial Intelligence* 2(3-4). Le formalisme original (préconditions + add/delete lists).
- Russell, S., & Norvig, P. — *Artificial Intelligence: A Modern Approach* (4e éd., 2021), ch. « Planning and Acting in the Real World ». Recherche avant, heuristiques de relaxation.
- Bonet, B., & Geffner, H. (2001) — « Planning as heuristic search », *Artificial Intelligence* 129(1-2). Les heuristiques delete-free qui font le cœur de `pyperplan`/`Fast Downward`.

## Navigation

[← Planners (parent)](../README.md) | [Planners-6 Python](Planners-6-Domains.ipynb) | [Planners-7 →](Planners-7-OR-Tools.ipynb)
